# Quick start - inbrain_dementia_data_analyser

From raw tables to a cohort: validation, pre-processing, integration and the
four cohort tools. Runs locally or in Google Colab. No data are bundled: set
`ROOT` to the folder holding your `Tables_<DATASET>` folders (see the README,
section 2).

## 0. Setup

In [ ]:
# Google Colab only (skip locally after `pip install -e .`)
# from google.colab import drive
# drive.mount("/content/drive")
# !git clone -q https://github.com/<your-org>/inbrain_dementia_data_analyser.git /content/inbrain_dementia_data_analyser
# !pip install -q /content/inbrain_dementia_data_analyser

import inbrain_dementia_data_analyser as ida

ida.enable_logging("INFO")
ROOT = "/content/drive/MyDrive/VRS_inbrain_documents/Tables_projects_dementia"   # <- your root folder
cfg = ida.load_config()          # or ida.load_config("/path/to/my/configs")
print(cfg.config_dir, list(cfg.datasets))

## 1. Validation
Checks folders, files (by pattern, any download date) and the columns each pipeline needs.

In [ ]:
reports = ida.validate_all(ROOT)
for rep in reports.values():
    print(rep.summary())
reports["ADNI"].to_frame()

## 2. Pre-processing
One table per dataset, written to `Tables_<DS>/Integrated_<DS>/`. Run the datasets you have; each is freed from memory before the next.

In [ ]:
summary = ida.preprocess_all(ROOT, ["AIBL", "NIFD", "ABVIB", "4RTNI", "MCSA", "A4",
                                    "PREVENT-AD", "OASIS", "ADNI", "NACC"])
summary

## 3. Integration
Merged master (harmonised names, demographics and diagnoses) and the expanded master (6-month windows around the first diagnosis).

In [ ]:
paths = ida.integrate(ROOT)
X = paths["expanded"]
paths

## 4. Cohorts
### Cross-sectional: anchor pairing (one event per subject)

In [ ]:
cs = ida.create_cross_sectional_cohort(
    X, method="anchor", anchor="MRI_T1w", required=["CDR", "CDR_SB", "BMI"], optional=["MRI_T2w"],
    margin_months=6, margin_months_by_feature={"BMI": 12}, anchor_events="most_optional",
    output_prefix="T1w_CDR_BMI")
print(cs.n_subjects, cs.files)
cs.table.head()

### Cross-sectional: multimodal set

In [ ]:
mm = ida.create_multimodal_cohort(X, window_months=12, output_prefix="T1_FDG", modalities=[
    {"label": "MRI_T1w", "prefixes": ["MRI_T1w"], "required": True},
    {"label": "PET_FDG", "prefixes": ["PET_FDG"], "required": True},
    {"label": "PET_ABETA", "prefixes": ["PET_ABETA"], "required": False},
])
mm.table["Databank"].value_counts()

### Longitudinal: one modality on a schedule

In [ ]:
lg = ida.create_longitudinal_cohort(X, modality_label="MRI_T1w", visit_targets_months=[0, 12, 24],
                                    tolerance_months=6, min_visits=2, output_prefix="T1w")
lg.table.filter(like="_matched").sum()

### Longitudinal multimodal: every visit carries a complete set

In [ ]:
lm = ida.create_longitudinal_multimodal_cohort(
    X, schedule_modality="MRI_T1w", required=["CDR", "PET_ABETA"], optional=["PET_FDG"],
    margin_months=12, visit_targets_months=[0, 24, 48], tolerance_months=9, min_visits=2,
    output_prefix="T1_CDR_amyloid")
lm.summary["n_visits_matched"].value_counts()

## 5. Image IDs for the LONI IDA search

In [ ]:
ids = lg.image_ids()
print(len(ids), ",".join(ids[:10]), "...")